In [2]:
# [CELL 1] PROFILE STUDENT CONFIGURATION
from pathlib import Path

LAST_NAME = "Dela Cruz"
STUDENT_ID = "TUPM-1251-26"
SEED_DIGIT = 6

# Data Generation Rules from the manual:
# Operating System: Android if SEED_DIGIT is even; iOS if odd
# User Behavior Class: (SEED_DIGIT % 5) + 1
ASSIGNED_OS = "Android"  # 6 is even
TARGET_BEHAVIOR_CLASS = (6 % 5) + 1  # 2

print("Assigned Operating System:", ASSIGNED_OS)
print("Target User Behavior Class:", TARGET_BEHAVIOR_CLASS)


Assigned Operating System: Android
Target User Behavior Class: 2


In [3]:
# [CELL 2] INITIALIZE WORKSPACE PATHS FOR PROFILE
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")

profile_dataset_path = DATA_DIR / "user_behavior_dataset.csv"
profile_csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_profile_filtered.csv"
json_summary_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_profile_summary.json"
profile_backup_path = BACKUP_DIR / f"{LAST_NAME}_{STUDENT_ID}_profile_filtered.csv"

print("Dataset path:", profile_dataset_path)
print("Filtered output path:", profile_csv_path)
print("JSON summary path:", json_summary_path)
print("Backup path:", profile_backup_path)


Dataset path: data\user_behavior_dataset.csv
Filtered output path: output\Dela Cruz_TUPM-1251-26_profile_filtered.csv
JSON summary path: output\Dela Cruz_TUPM-1251-26_profile_summary.json
Backup path: backup\Dela Cruz_TUPM-1251-26_profile_filtered.csv


In [4]:
# [CELL 3] READ AND FILTER MOBILE DEVICE DATASET
total_records = 0
matching_os_records = 0
target_behavior_records = 0

total_battery_drain = 0.0
total_data_usage = 0.0
selected_users_data = []

with open(profile_dataset_path, "r", encoding="utf-8") as file:
    header = file.readline().strip().split(",")
    
    # Get column indices based on required exported fields and criteria
    id_idx = header.index("User ID")
    model_idx = header.index("Device Model")
    os_idx = header.index("Operating System")
    battery_idx = header.index("Battery Drain (mAh/day)")
    data_idx = header.index("Data Usage (MB/day)")
    behavior_idx = header.index("User Behavior Class")
    
    for line in file:
        fields = line.strip().split(",")
        total_records += 1
        
        current_os = fields[os_idx]
        current_behavior = int(fields[behavior_idx])
        
        # Check first criteria: Match Assigned Operating System
        if current_os == ASSIGNED_OS:
            matching_os_records += 1
            
            # Check second criteria: Match Target Behavior Class
            if current_behavior == TARGET_BEHAVIOR_CLASS:
                target_behavior_records += 1
                
                user_id = fields[id_idx]
                device_model = fields[model_idx]
                battery_drain = float(fields[battery_idx])
                data_usage = float(fields[data_idx])
                
                # Accumulate values for averages calculation
                total_battery_drain += battery_drain
                total_data_usage += data_usage
                
                # Save specified columns for CSV export
                selected_users_data.append([user_id, device_model, current_os, fields[battery_idx], fields[data_idx]])

print("Total rows evaluated:", total_records)
print("Rows matching OS:", matching_os_records)
print("Rows matching target behavior class:", target_behavior_records)


Total rows evaluated: 700
Rows matching OS: 554
Rows matching target behavior class: 114


In [5]:
# [CELL 4] CALCULATE METRIC AVERAGES
if target_behavior_records > 0:
    average_battery_drain = total_battery_drain / target_behavior_records
    average_data_usage = total_data_usage / target_behavior_records
else:
    average_battery_drain = 0.0
    average_data_usage = 0.0

print("Calculated Metric Averages:")
print("Average Battery Drain (mAh/day):", average_battery_drain)
print("Average Data Usage (MB/day):", average_data_usage)


Calculated Metric Averages:
Average Battery Drain (mAh/day): 887.1140350877193
Average Data Usage (MB/day): 451.43859649122805


In [6]:
# [CELL 5] EXPORT FILTERED USERS TO CSV AND SUMMARY TO JSON
import json

# Export columns to filtered CSV file
with open(profile_csv_path, "w", encoding="utf-8") as out_csv:
    out_csv.write("User ID,Device Model,Operating System,Battery Drain (mAh/day),Data Usage (MB/day)\n")
    for row in selected_users_data:
        out_csv.write(f"{row[0]},{row[1]},{row[2]},{row[3]},{row[4]}\n")

# Prepare configuration logging block dictionary
summary_data = {
    "student_last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "operating_system": ASSIGNED_OS,
    "user_behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_records_in_dataset": total_records,
    "records_matching_os": matching_os_records,
    "records_in_target_behavior": target_behavior_records,
    "average_battery_drain_mah": average_battery_drain,
    "average_data_usage_mb": average_data_usage,
    "row_count_verified": len(selected_users_data) == target_behavior_records
}

with open(json_summary_path, "w", encoding="utf-8") as out_json:
    json.dump(summary_data, out_json, indent=4)

print("Filtered Profile CSV created:", profile_csv_path.exists())
print("JSON Summary file created:", json_summary_path.exists())


Filtered Profile CSV created: True
JSON Summary file created: True


In [7]:
# [CELL 6] CREATE BACKUP AND REPORT FILE SIZE METADATA
import shutil

# Create backup of the exported CSV file as requested by Requirement 6
shutil.copy2(profile_csv_path, profile_backup_path)

backup_info = profile_backup_path.stat()

print("--- Final Profile Assessment Results ---")
print("Operating system and behavior class:", f"{ASSIGNED_OS} - Class {TARGET_BEHAVIOR_CLASS}")
print("Total records in dataset:", total_records)
print("Records matching operating system:", matching_os_records)
print("Records in target behavior class:", target_behavior_records)
print("Average battery drain (mAh/day):", round(average_battery_drain, 2))
print("Average data usage (MB/day):", round(average_data_usage, 2))
print("Count verification (Match):", len(selected_users_data) == target_behavior_records)
print("Exported CSV file:", profile_csv_path.name)
print("JSON summary file:", json_summary_path.name)
print("Backup file size:", backup_info.st_size, "bytes")


--- Final Profile Assessment Results ---
Operating system and behavior class: Android - Class 2
Total records in dataset: 700
Records matching operating system: 554
Records in target behavior class: 114
Average battery drain (mAh/day): 887.11
Average data usage (MB/day): 451.44
Count verification (Match): True
Exported CSV file: Dela Cruz_TUPM-1251-26_profile_filtered.csv
JSON summary file: Dela Cruz_TUPM-1251-26_profile_summary.json
Backup file size: 4117 bytes
